# Lorde Forge · Kaggle runbook (Phase 0)

Runs the whole pipeline on a **Kaggle 2×T4** notebook (Internet on, GPU T4 x2). Three sessions:

| Session | Steps | Envs | Time |
|---|---|---|---|
| A · data | 0-3: serve teacher/student, build pool, teacher labels, splits, golden set | serve, data | 2-3 h |
| B · train | 5: LoRA SFT with AlignTune, merge | train | 4-6 h |
| C · evaluate | 6-9: judge eval, CuratorKIT plug-in runs, downstream test, bench, publish | serve, data, train | 4-6 h |

**Every command comes from `forge/kaggle_driver.py`**, the headless driver that produced the published run:
`step(session, name)` prints the driver's script for that step and runs it, so this notebook and the driver
cannot drift apart (the downstream step, for instance, uses the driver's `DOWNSTREAM_ARGS`). The same sessions
run without a notebook UI as `python forge/kaggle_driver.py a|b|c|d`.

Step 4 (hand labels) runs on your laptop or a small hosted service: see the Step 4 cell below.

**Secrets** (Add-ons → Secrets): `HF_TOKEN` (read access to the gated Cohere models; write scope for publishing),
`COHERE_API_KEY` (32B calibration slice, optional). Accept the licences on the Tiny Aya Global, Tiny Aya Base and
Aya Expanse 8B model pages first.

`data/`, `results/` and `bench/` are synced to a **private** HF dataset (`<you>/lorde-work`) after each step, with
the driver's own rules (human labels are never uploaded from here), so a killed session loses nothing: every step
resumes from its cache. In a new session, run the four set-up cells again first.

In [ ]:
# ---- configure: environment variables win; the defaults are this project's ----
import os

HF_USER = os.environ.get("HF_USER", "Oxlino")
REPO = os.environ.get("LORDE_REPO", "https://github.com/linoxbt/Lorde.git")
BRANCH = os.environ.get("LORDE_BRANCH", "main")
WORK_REPO = os.environ.get("WORK_REPO", f"{HF_USER}/lorde-work")   # private dataset: session-to-session cache
os.environ.update(HF_USER=HF_USER, WORK_REPO=WORK_REPO)
os.environ.setdefault("HUMAN_LANGS", "en,fr,es")   # languages YOU can read well enough to hand-label (step 3)
# Optional: a separate private repo for the human-sample key, one the label service's token cannot read (S6).
# os.environ["LORDE_LABEL_KEY_REPO"] = f"{HF_USER}/lorde-label-key"

In [ ]:
from kaggle_secrets import UserSecretsClient

s = UserSecretsClient()
os.environ["HF_TOKEN"] = s.get_secret("HF_TOKEN")
try:
    os.environ["COHERE_API_KEY"] = s.get_secret("COHERE_API_KEY")
except Exception:
    print("no COHERE_API_KEY: the 32B calibration slice will be skipped")
os.environ["VLLM_API_KEY"] = "lorde-local"
os.environ["LITELLM_LOG"] = "ERROR"

In [ ]:
%%bash -s "$REPO" "$BRANCH"
cd /kaggle/working
[ -d Lorde ] || git clone -b "$2" "$1" Lorde
cd Lorde && git fetch -q origin "$2" && git checkout -q "$2" && git pull -q && git log --oneline | head -3
pip install -q -U uv

In [ ]:
# Load the driver: its step scripts, its Hub sync rules and its checks.
import importlib.util
import subprocess

os.chdir("/kaggle/working/Lorde")
_spec = importlib.util.spec_from_file_location("kaggle_driver", "forge/kaggle_driver.py")
drv = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(drv)


def step(session: str, name: str) -> None:
    """Run one step of forge/kaggle_driver.py exactly as the driver would (same script, same bash flags)."""
    script = dict(drv.SESSIONS[session])[name]
    script = script() if callable(script) else script
    print(f"# [{session}] {name}\n{script}\n", flush=True)
    subprocess.run(["bash", "-c", "set -euo pipefail\ncd " + str(drv.ROOT) + "\n" + script], check=True)


hub = drv.Hub()          # creates the private work repo if needed
hub.pull()               # data/, results/, bench/, merged model and train_meta.json from earlier sessions


def sync(human_sample: bool = False) -> None:
    """Upload data/, results/ and bench/ with the driver's ignore rules (never human labels)."""
    hub.push(("data", "results", "bench", "logs"), message="runbook sync", human_sample=human_sample)


# Three separate environments (spec 4.1) in /tmp, clear of Kaggle's 20 GB output limit; checks 2 GPUs.
step("a", "setup")

## Session A · data (steps 0-3)

**Smoke-test vLLM on T4 first.** T4 is compute capability 7.5 (no bf16): `00_serve.sh` switches to fp16 by itself.
If the teacher fails to start, read `logs/vllm_teacher.log`; `VLLM_EXTRA_ARGS="--enforce-eager"` or
`GPU_UTIL=0.85` are the usual fixes.

In [ ]:
step("a", "pool-teacher")
sync()

In [ ]:
step("a", "pool-student")   # student answers, then the assembled pool
sync()

In [ ]:
step("a", "teacher-label")
sync()
ok, note = drv.exit_check_a()
print("EXIT CHECK PASSED: continue" if ok else "EXIT CHECK FAILED: stop here and share the summary", "|", note)

In [ ]:
step("a", "splits")         # splits, the blind human sample and its private key, Bench sets
sync(human_sample=True)     # the only sync that may upload the human sample (and the key, see LORDE_LABEL_KEY_REPO)

In [ ]:
step("a", "calibration-32b")   # Aya Expanse 32B on golden-300 via the Cohere API; skipped without COHERE_API_KEY
sync()

## Step 4 · human labels (laptop or a hosted label service)

```bash
git clone -b "${LORDE_BRANCH:-main}" "${LORDE_REPO:-https://github.com/linoxbt/Lorde.git}" Lorde && cd Lorde && pip install -r requirements/forge-data.txt
export HF_TOKEN=hf_...   # a token with write access to the work repo only
python forge/04b_blind_sample.py --hub "${WORK_REPO:-$HF_USER/lorde-work}"   # once: blind sample + private key (no-op if done)
# Blind labelling, synced with the work repo (pulls the sample, merges hub + local labels, pushes every save with
# a compare-and-swap commit). A password is required whenever the page is reachable beyond localhost.
LABEL_PASSWORD=... LABEL_NAME=<you> python forge/04_label_tool.py --hub "${WORK_REPO:-$HF_USER/lorde-work}" --share   # 200 items
# Never upload data/human_labels.jsonl by hand: the tool merges by (item, labeller) and newest labeled_at.
```

**Keep the labels independent:** do not look items up in the public `lorde-data` dataset (its pool and test files
name each item's variant) while labelling, and keep the sample key where the label token cannot read it
(`LORDE_LABEL_KEY_REPO`, see `forge/04_label_tool.py`).

## Session B · train (step 5)

The driver times a 160-row smoke run first and picks the epochs that fit `TRAIN_BUDGET_H` (or set `TRAIN_ARGS`).
If AlignTune misbehaves with Tiny Aya, set `TRAIN_ARGS="--engine trl --epochs 2"`.

In [ ]:
hub.pull(checkpoints=True)   # resume from the newest uploaded checkpoint of an interrupted run
step("b", "train-smoke")

In [ ]:
step("b", "train")
print(drv.save_model(hub))   # merged model + train_meta.json to the work repo, for Session C

## Session C · evaluate (steps 6-9)

In [ ]:
hub.pull_human_labels()      # the label tool's latest labels (and the key) before step 6 reads them
step("c", "judge-eval")      # step 6 (raw + structured), Bench, plug-in runs with base and Lorde
sync()

In [ ]:
step("c", "plugin-teacher")
sync()

In [ ]:
# Downstream: three small SFTs (ungated / teacher-gated / Lorde-gated, one per GPU), answers, AuditKIT scores.
print("DOWNSTREAM_ARGS:", drv.DOWNSTREAM_ARGS)   # the settings behind the published downstream numbers
step("c", "downstream")
sync()

In [ ]:
hub.pull_human_labels()
step("c", "publish-private")   # private: review everything on the Hub first
# then make it public (un-gated):
# subprocess.run(["bash", "-c", f"envs/data/bin/python forge/09_publish.py --user {HF_USER} --public"], check=True)

### Re-running step 6 predictions (no legacy caches)

The published judge predictions (`results/preds/*.jsonl`) were computed in Session C and reused in Session D with
`--trust-legacy-cache`, which stamps them `legacy_trusted: true` in `results/preds/*.meta.json`: nothing proves
Session C served the same weights. The driver no longer passes that flag unless `LORDE_TRUST_LEGACY_CACHE=1`, so
this cell recomputes them (937 items per judge; the old cache is moved aside to `*.jsonl.stale.*`):

In [ ]:
os.environ.pop("LORDE_TRUST_LEGACY_CACHE", None)
_spec.loader.exec_module(drv)   # reload: TRUST_LEGACY is read at import
step("d", "judge-timing")       # fresh timings and fresh, fingerprinted predictions for base and Lorde
sync()

### Optional: serve trained Lorde on the live site (CPU)

Exports the merged model to GGUF (Q8_0, about 3.6 GB) with llama.cpp's converter and pushes it to
`<you>/lorde-tiny-aya-GGUF`. Then set `LLAMA_ARG_HF_REPO=<you>/lorde-tiny-aya-GGUF:Q8_0` on the Railway
`lorde-serve` service, and `LORDE_MODEL_REPO=lorde-tiny-aya`, `LORDE_MODEL_REVISION=v1.0.0` on `lorde-api`.

In [ ]:
%%bash -s "$HF_USER"
cd /kaggle/working/Lorde
envs/train/bin/python forge/10_export_gguf.py --user "$1"          # add --public when you publish the rest